## Import Libraries

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv



MODEL = "BAAI/bge-base-en-v1.5"
EMBEDDING_SIZE = 768

d:\0. Projects\3. Agentic Gen\Vidzy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\0. Projects\3. Agentic Gen\Vidzy\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


## Step 1 — Fetch YouTube Transcript

In [ ]:
video_id = input("Paste the Video ID here: ").strip()

try:
    api = YouTubeTranscriptApi()
    transcript = api.fetch(video_id, languages=["en", "hi", "es", "fr", "de"])
    transcript_list = transcript.to_raw_data()
    transcript_text = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript_text[:500], "...")

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"Error: {e}")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful  ...


## Step 2 — Split into Chunks

In [3]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_text])
print(f"Total chunks: {len(chunks)}")

Total chunks: 168


## Step 3 — Generate Embeddings & Store in Vector DB

In [5]:
# embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
# embeddings = FastEmbedEmbeddings(model_name=MODEL)

load_dotenv()
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
)

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
print("Vector store ready!")

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://router.huggingface.co/hf-inference/models/sentence-transformers/all-MiniLM-L6-v2/pipeline/feature-extraction' (Request ID: Root=1-6ad8924c-4161b5477d87e77371efc715;92e91d46-2aa2-47f6-9909-989a21e5f1dc)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.

## Step 4 — Build the RAG Chain

In [ ]:
config = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=HF_API_TOKEN,
)

llm = ChatHuggingFace(llm=config)


prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know no context in the video.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

def format_docs(retrieved_docs):
    return "\n\n".join(doc.page_content for doc in retrieved_docs)

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

parser = StrOutputParser()

main_chain = parallel_chain | prompt | llm | parser
print("Chain ready!")

## Step 5 — Ask a Question

In [ ]:
question = input("Enter your question: ")
answer = main_chain.invoke(question)
print(answer)